In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import pip
! pip install matplotlib
! pip install seaborn
! pip install scipy
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro

## Анализ цен акций

In [ ]:
df = pd.read_csv('quotes_all.csv')
df

In [ ]:
df = df.drop(columns=['Unnamed: 0'])
df['Date'] = pd.to_datetime(df['Date'])
df['Volume'] = df['Volume'].astype(int)
df.info()

In [ ]:
df.drop_duplicates
df.shape[0]

In [ ]:
df.isna().sum()

В датафрейме нет пропусков и дубликатов, он готов к работе.

Добавим delta - столбец отношения close date текущего и предыдущего дня. Для первого дня заполним потенциальный пропуск нулем.

In [ ]:
df['delta'] = df.groupby('ticker')['Close'].transform( lambda x: x / x.shift(1)).fillna(0)

Логарифмируем его, возможно это пригодится далее для анализа:

In [ ]:
df['ln_delta'] = np.log(df['delta'].replace(0, np.nan)).fillna(0)

In [ ]:
df

In [ ]:
df['ticker'].value_counts()

In [ ]:
df_aapl  = df[df['ticker'] == 'AAPL']
df_xom  = df[df['ticker'] == 'XOM']
df_tsla = df[df['ticker'] == 'TSLA']
df_wmt = df[df['ticker'] == 'WMT']
df_pfe = df[df['ticker'] == 'PFE']
df_nflx = df[df['ticker'] == 'NFLX']
df_jpm = df[df['ticker'] == 'JPM']
df_cat = df[df['ticker'] == 'CAT']

ticker_dfs = [df_aapl, df_xom, df_tsla, df_wmt, df_pfe, df_nflx, df_jpm, df_cat]

In [ ]:
for dft in ticker_dfs:
    print("\n")
    print(f"Компания: {dft['ticker'].iloc[0]}")
    print(dft.describe())

In [ ]:
for dft in ticker_dfs:
    data = dft['Date']

In [ ]:
df['Year'] = df['Date'].dt.year

pivot_table = pd.crosstab(df['ticker'], df['Year'])

pivot_table

Число наблюдений совпадает для всех компаний, что подтверждает отсутствие ошибок при сборе данных.

# Univariate analysis

In [ ]:
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'delta', 'ln_delta']

Посмотрим на распределение значений во всех числовых колонках для акций всех компаний по-отдельности.

In [ ]:
for dft in ticker_dfs:
    print(f"Компания: {dft['ticker'].iloc[0]}")
    for col in numeric_cols:
        fig, axes = plt.subplots(1, 3, figsize=(20, 4))
        fig.suptitle(f'{col}', fontsize=14)
        sns.histplot(dft[col], ax=axes[0])
        axes[0].set_title('Гистограмма')
        sns.boxplot(x=dft[col], ax=axes[1])
        axes[1].set_title('Ящик с усами')
        sns.violinplot(x=dft[col], ax=axes[2])
        axes[2].set_title('Скрипичная диаграмма')
        plt.tight_layout()
        plt.show()

Заметим что:
1) Open, High, Low, Close, Adj Close в рамках одной компании практически идентичны
2) У распределения цен имеется 2-3 пика
3) Распределение delta визуально близко к нормальному, проверим, так ли это:

In [ ]:
for dft in ticker_dfs:
    print("\n")
    print(f"Компания: {dft['ticker'].iloc[0]}")

    stat_delta, p_delta = shapiro(dft['delta'].values)
    print(f"delta: значение статистики = {stat_delta:.4f}, p-value={p_delta:.4f}")
    if p_delta > 0.05:
        print("Распределение статистически не отличается от нормального (p > 0.05)")
    else:
        print("Распределение не является нормальным (p <= 0.05)")

    stat_ln, p_ln = shapiro(dft['ln_delta'].values)
    print(f"ln_delta: значение статистики = {stat_ln:.4f}, p-value={p_ln:.4f}")
    if p_ln > 0.05:
        print("Распределение статистически не отличается от нормального (p > 0.05)")
    else:
        print("Распределение не является нормальным (p <= 0.05)")

Убедимся, что на более маленьких случайных подвыборках результаты будут аналогичными:

In [ ]:
for dft in ticker_dfs:
    print("\n")
    print(f"Компания: {dft['ticker'].iloc[0]}")

    stat_delta, p_delta = shapiro(dft['delta'].sample(100, random_state=69).values)
    print(f"delta: значение статистики = {stat_delta:.4f}, p-value={p_delta:.4f}")
    if p_delta > 0.05:
        print("Распределение статистически не отличается от нормального (p > 0.05)")
    else:
        print("Распределение не является нормальным (p <= 0.05)")

    stat_ln, p_ln = shapiro(dft['ln_delta'].sample(100, random_state=69).values)
    print(f"ln_delta: значение статистики = {stat_ln:.4f}, p-value={p_ln:.4f}")
    if p_ln > 0.05:
        print("Распределение статистически не отличается от нормального (p > 0.05)")
    else:
        print("Распределение не является нормальным (p <= 0.05)")


Таким образом, это все же не нормальное распределение.

Те же визуализации для общего датасета по всем компаниям:

In [ ]:
for col in numeric_cols:
    fig, axes = plt.subplots(1, 3, figsize=(20, 4))
    fig.suptitle(f'{col}', fontsize=14)
    sns.histplot(df[col], ax=axes[0])
    axes[0].set_title('Гистограмма')
    sns.boxplot(x=df[col], ax=axes[1])
    axes[1].set_title('Ящик с усами')
    sns.violinplot(x=df[col], ax=axes[2])
    axes[2].set_title('Скрипичная диаграмма')
    plt.tight_layout()
    plt.show()

Графики демонстируют, что в общем датасете содержится смесь распределений для акций разных компаний.

В общих данных и в данных по компаниям по-отдельности есть много выбросов, однако удалять их нет смысла, так как они представляют интерес в рамках поставленной задачи - модель должна прелсказывать в том числе и резкие колебания цен, на которых в первую очередь можно заработать.

# Multivariate analysis

In [ ]:
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(7, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', linewidths=0.5)
plt.title('Тепловая карта корреляций для числовых столбцов')
plt.tight_layout()
plt.show()

Как было сказано ранее, столбцы Open, High, Low, Close, Adj Close практически идентичны (т.к. изменения цены в рамках 1 дня незначительны). Переменная volume не имеет значительной корреляции ни с одной другой переменной. Аналогичная ситуация наблюдается для таргета delta и ln_delta, однако на данном этапе это не плохо, т.к. мы будем предсказывать изменение цены не по информации о текущих ценах, а по новостям из другого датасета.

Визуализируем попарные диаграммы рассеяния:

In [ ]:
sns.pairplot(df[numeric_cols])
plt.show()

Убедились, что никаких сложных нелинейных зависимостей мы не упустили.

In [ ]:
quotes = df.copy()

## Анализ новостей

In [ ]:
df = pd.read_csv('data_model.csv')
df

Первые 8 колонок - OHE издания, выпустившего новость
Следующие 8 колонок - тикеры компаний, указанных издательством в новости
Далее дата публикации, далее столбей со списком компаний для которых новость была найдена 
Дальше идут эмбединги: s - эмбединг для короткого содержания, f - эмбединг для полного текста новости.

In [ ]:
df.info()

In [ ]:
df.isna().sum().sum()

Т.к. эмбединги полного текста построены не для всех новостей, датасет был подразбит:

In [ ]:
data_s = pd.read_csv('data_s.csv')
data_s.drop_duplicates
data_s

In [ ]:
data_s.isna().sum().sum()

In [ ]:
data_f = pd.read_csv('data_f.csv')
data_f.drop_duplicates
data_f

In [ ]:
data_f.isna().sum().sum()

В обоих датафреймах нет (и не было после изначальной обработки на этапе сбора) дубликатов, отсутствуют пропуски.

## Univariate analysis

In [ ]:
publishers = ['Benzinga', 'CNBC', 'ChartMill', 'DowJones', 'MarketWatch', 'SeekingAlpha', 'The Guardian', 'Yahoo']

def tmp(df, publishers):
    counts = df[publishers].sum()
    return (counts / counts.sum() * 100).round(2)

perc_df = pd.DataFrame({
    'Издание': publishers,
    'data_s': tmp(data_s, publishers).values,
    'data_f': tmp(data_f, publishers).values})

perc_df = perc_df.sort_values('data_s', ascending=False)
perc_df

В data_s вошли преимущественно новости из The Guardian и Yahoo, в data_f - только из The Guardian

In [ ]:
tickers = ['AAPL', 'TSLA', 'CAT', 'JPM', 'NFLX', 'PFE', 'XOM', 'WMT']

def tmp(df, tickers):
    counts = df[tickers].sum()
    counts = counts[counts > 0]
    counts = counts.sort_values(ascending=False)
    return counts

counts_s = tmp(data_s, tickers)
counts_f = tmp(data_f, tickers)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.pie(counts_s, labels=counts_s.index, autopct='%1.1f%%')
ax1.set_title('Распределение новостей по компаниям в data_s')

ax2.pie(counts_f, labels=counts_f.index, autopct='%1.1f%%')
ax2.set_title('Распределение новостей по компаниям в data_f')

plt.tight_layout()
plt.show()

В data_f тикеры упоминаются практически равновероятно, в data_f заметно чаще других встречается JPMorgan.

In [ ]:
company_names = ["Apple", "Tesla", "Caterpillar", "JPMorgan", "Netflix", "Pfizer", "Exxon", "Walmart"]

def tmp(df, names):
    percentages = {}
    for name in names:
        count = df['company'].str.contains(name).sum()
        percentages[name] = (count / len(df)) * 100
    return percentages

pct_s = pd.Series(tmp(data_s, company_names))
pct_f = pd.Series(tmp(data_f, company_names))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
x = np.arange(8)

bars1 = ax1.bar(x, pct_s[company_names])
ax1.set_title('Упоминания компаний в data_s')
ax1.set_ylabel('% новостей, относящихся к компании')
ax1.set_ylim(0, 30)
ax1.set_xticks(x)
ax1.set_xticklabels(company_names, rotation=45)


bars2 = ax2.bar(x, pct_f[company_names])
ax2.set_title('Упоминания компаний в data_f')
ax2.set_ylabel('% новостей, относящихся к компании')
ax2.set_ylim(0, 30)
ax2.set_xticks(x)
ax2.set_xticklabels(company_names, rotation=45)


plt.tight_layout()
plt.show()

В среднем для компаний оказываются релевантными 15% до 26% статей из датасетов, значит мы собрали информативные данные.